# Lab C11.1–C11.2 · Czas i dostępność cech

Przed danym etapem przeczytaj wskazany wykład. Notebooki uruchamiaj ze wspólnego folderu `labs`, zgodnie z głównym README. Pracuj we własnej kopii repozytorium.

Przewiduj wyniki przed wykonaniem kodu. Nie musisz zapisywać odpowiedzi na pytania do przemyślenia. Sprawdzenia pustych starterów celowo nie przechodzą.

In [ ]:
from pathlib import Path
import sys
assert sys.version_info[:2] == (3, 12), "Wybierz kernel Pythona 3.12 z głównego README."
lab_dir = Path.cwd()
assert (lab_dir / "lab-c11-1-c11-2.ipynb").is_file(), "Uruchom Jupyter z folderu labs."
print("Folder:", lab_dir)

In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError as error:
    raise AssertionError("Brakuje bibliotek sekcji C. Dodaj je w practice i uruchom Jupyter przez --project ../practice, zgodnie z głównym README.") from error
print("pandas", pd.__version__, "| numpy", np.__version__)

## Ten sam panel po decyzji z C10

Gotowa komórka wczytuje panel trzech produktów i wykonuje decyzję z labu C10.1–C10.3: usuwa jedną kopię wiersza, a brak sprzedaży pozostawia. Wiersze nadal są w przypadkowej kolejności. Pełny zakres dni to 1–6.

In [ ]:
panel = pd.read_csv(lab_dir / "data" / "c-panel-sales.csv")
clean = panel.drop_duplicates(["item", "day"])
assert len(clean) == 17, "Oczekiwano 17 wierszy po usunięciu jednej kopii."
full_days = list(range(1, 7))
print(clean.head())

## C11.1 · Najpierw porządek, potem szukanie luk

Po wykładzie C11.1 zapisz w `ordered` tabelę `clean` posortowaną po `item` i `day`, z indeksem od zera. W `missing_days` zapisz słownik: dla każdego produktu lista dni z zakresu `full_days`, których brakuje w jego serii. Przewidź, który produkt ma lukę.

In [ ]:
ordered = None
missing_days = None
# sort_values po dwóch kolumnach i reset_index(drop=True). Dni serii porównaj ze zbiorem full_days.

In [ ]:
assert isinstance(ordered, pd.DataFrame), "C11.1: ordered ma być tabelą."
assert ordered["item"].tolist() == ['apple', 'apple', 'apple', 'apple', 'apple', 'pear', 'pear', 'pear', 'pear', 'pear', 'pear', 'plum', 'plum', 'plum', 'plum', 'plum', 'plum'], "C11.1: sortuj najpierw po item, potem po day."
assert ordered["day"].tolist() == [1, 2, 3, 4, 6, 1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6], "C11.1: dni mają rosnąć wewnątrz każdej serii."
assert ordered.index.tolist() == list(range(17)), "C11.1: po sortowaniu użyj reset_index(drop=True)."
assert missing_days == {"apple": [5], "pear": [], "plum": []}, "C11.1: porównaj dni każdej serii z full_days; brak sprzedaży plum to nie brak dnia."
print("C11.1: serie są uporządkowane, a luka w dniach jest nazwana.")

<details><summary>Podpowiedź 1</summary>

sort_values przyjmuje listę kolumn. Bez reset_index zostają stare etykiety.

</details>

<details><summary>Podpowiedź 2</summary>

Dla jednej serii wybierz jej dni maską po item, zamień na zbiór i odejmij od zbioru full_days. Pętla po trzech produktach wypełni słownik.

</details>

## C11.1 · Funkcja lag w pliku practice/features.py

Przesunięcie wewnątrz serii przyda się w kolejnych labach, dlatego trafia do pliku. W edytorze otwórz `practice/features.py` i uzupełnij `lag_within_series(table, key, column, periods)` według docstringa. Funkcja zwraca Series o indeksie tabeli: wartości `column` przesunięte o `periods` wierszy wstecz, osobno dla każdej wartości `key`. Pierwsze `periods` wierszy każdej serii są brakiem. Tabela wejściowa pozostaje bez zmian. Zachowaj sygnaturę i adnotacje typów.

Po zapisaniu pliku wykonaj gotową komórkę poniżej; wczytuje plik ponownie, jak w labie B6.1–B6.2.

In [ ]:
import runpy
features_path = lab_dir.parent / "practice" / "features.py"
assert features_path.is_file(), "Brakuje practice/features.py w repozytorium."
lag_within_series = runpy.run_path(str(features_path))["lag_within_series"]

In [ ]:
tiny = pd.DataFrame({"item": ["a", "a", "a", "b", "b"], "day": [1, 2, 3, 1, 2], "sale": [2.0, 4.0, 8.0, 20.0, 40.0]})
tiny_before = tiny.copy()
tiny_lag1 = lag_within_series(tiny, "item", "sale", 1)
assert isinstance(tiny_lag1, pd.Series) and tiny_lag1.index.tolist() == tiny.index.tolist(), "C11.1: zwróć Series o indeksie tabeli, nie całą tabelę ani pustą serię."
assert tiny_lag1.isna().tolist() == [True, False, False, True, False], "C11.1: pierwszy wiersz każdej serii ma być brakiem; seria b nie widzi serii a."
assert tiny_lag1.tolist()[1:3] == [2.0, 4.0] and tiny_lag1.tolist()[4] == 20.0, "C11.1: przesuń o jeden wiersz wstecz wewnątrz serii."
tiny_lag2 = lag_within_series(tiny, "item", "sale", 2)
assert tiny_lag2.isna().tolist() == [True, True, False, True, True] and tiny_lag2.tolist()[2] == 2.0, "C11.1: periods=2 cofa o dwa wiersze; dwudniowa seria ma same braki."
assert tiny.equals(tiny_before), "C11.1: funkcja nie dopisuje kolumny do tabeli wejściowej."
print("C11.1: lag jest liczony osobno dla każdej serii.")

<details><summary>Podpowiedź 1</summary>

Przykład wykładu C11.1 łączy groupby po kluczu z shift. Argument shift to liczba wierszy.

</details>

<details><summary>Podpowiedź 2</summary>

Wybierz kolumnę po groupby: table.groupby(key)[column]. Wynik shift ma już indeks tabeli; wystarczy go zwrócić.

</details>

## C11.1 · Lag i okno na uporządkowanym panelu

Zastosuj przesunięcie do `ordered`. W `lag1` zapisz poprzednią sprzedaż każdej serii: użyj swojej funkcji albo groupby i shift. W `mean2` zapisz średnią dwóch poprzednich wierszy serii, bez bieżącego dnia, jak w przykładzie `previous_mean` z wykładu. Potem obejrzyj wiersz apple z dnia 6 i zapisz w `apple_day6_lag_from_day`, z którego dnia pochodzi jego lag1.

In [ ]:
lag1 = None
mean2 = None
apple_day6_lag_from_day = None

In [ ]:
assert isinstance(lag1, pd.Series) and len(lag1) == len(ordered), "C11.1: lag1 ma jedną wartość na wiersz ordered."
assert lag1.isna().tolist() == [np.True_, np.False_, np.False_, np.False_, np.False_, np.True_, np.False_, np.False_, np.False_, np.False_, np.False_, np.True_, np.False_, np.False_, np.False_, np.True_, np.False_], "C11.1: pierwszy wiersz każdej serii jest brakiem; brak sprzedaży plum też przesuwa się jako brak."
assert lag1.iloc[4] == 5.0 and lag1.iloc[6] == 10.0, "C11.1: lag1 to poprzedni wiersz tej samej serii."
assert isinstance(mean2, pd.Series) and len(mean2) == len(ordered), "C11.1: mean2 ma jedną wartość na wiersz ordered; użyj transform."
assert mean2.isna().tolist() == [np.True_, np.True_, np.False_, np.False_, np.False_, np.True_, np.True_, np.False_, np.False_, np.False_, np.False_, np.True_, np.True_, np.False_, np.False_, np.True_, np.True_], "C11.1: dwa pierwsze wiersze serii i okna z brakiem dają brak."
assert mean2.iloc[2] == 2.0 and mean2.iloc[10] == 9.5, "C11.1: najpierw shift(1), potem rolling(2).mean(); bieżący dzień nie wchodzi do okna."
assert apple_day6_lag_from_day == 4, "C11.1: apple nie ma dnia 5, więc poprzedni wiersz dnia 6 pochodzi z dnia 4."
print("C11.1: przesunięcia nie mieszają serii, a luka zmienia znaczenie lag1.")

<details><summary>Podpowiedź 1</summary>

Bez groupby pierwszy wiersz pear otrzymałby ostatnią sprzedaż apple.

</details>

<details><summary>Podpowiedź 2</summary>

Dla mean2 napisz funkcję jak previous_mean z wykładu i przekaż ją do transform. Wiersz apple dnia 6 ma pozycję 4.

</details>

## C11.2 · Co jest dostępne wieczorem dnia 6

Po wykładzie C11.2 przygotowujesz prognozę na dni 7–10 wieczorem dnia 6. W `available` zbuduj tabelę z kolumnami `day`, `lag1`, `lag3`, `lag7` i `weekday`. Dla każdego dnia horyzontu wpisz True, gdy cecha jest dostępna w chwili prognozy: dzień źródłowy laga nie może wypadać po `origin`, a dzień tygodnia przyszłej daty jest znany zawsze. Najpierw wypełnij tabelę na kartce.

In [ ]:
origin = 6
horizon_days = [7, 8, 9, 10]
available = None
# pd.DataFrame z listami True/False według reguły: day - k <= origin.

In [ ]:
assert isinstance(available, pd.DataFrame), "C11.2: available ma być tabelą."
assert list(available.columns) == ["day", "lag1", "lag3", "lag7", "weekday"], "C11.2: kolumny day, lag1, lag3, lag7, weekday w tej kolejności."
assert available["day"].tolist() == horizon_days, "C11.2: jeden wiersz na każdy dzień horyzontu."
assert available["lag1"].tolist() == [True, False, False, False], "C11.2: lag1 dla dnia 8 wymaga sprzedaży z dnia 7, której wieczorem dnia 6 nie znasz."
assert available["lag3"].tolist() == [True, True, True, False], "C11.2: lag3 dla dnia 10 pochodzi z dnia 7; sprawdź day - 3 <= origin."
assert available["lag7"].tolist() == [True, True, True, True], "C11.2: lag7 dla dni 7–10 sięga dni sprzed origin, czyli wyłącznie historii."
assert available["weekday"].tolist() == [True, True, True, True], "C11.2: dzień tygodnia przyszłej daty jest znany z kalendarza."
print("C11.2: tabela dostępności odróżnia historię od przyszłości dla każdego dnia horyzontu.")

<details><summary>Podpowiedź 1</summary>

Dla każdego dnia odejmij długość laga i porównaj z origin. Comprehension po horizon_days daje gotową listę.

</details>

<details><summary>Podpowiedź 2</summary>

Słownik list przekazany do pd.DataFrame tworzy tabelę; kolejność kluczy decyduje o kolejności kolumn.

</details>

## C11.1 · Własne testy izolacji serii

Otwórz `practice/test_features.py`. Trzy funkcje testowe mają placeholdery do zastąpienia własnymi testami, tak jak w labie B6.1–B6.2. Na początku pliku dodaj `from features import lag_within_series`. Zbuduj w testach małą tabelę dwóch serii i sprawdź: drugi produkt nie widzi sprzedaży pierwszego, pierwszy wiersz każdej serii jest brakiem, a tabela po wywołaniu nie ma nowej kolumny. Do braków przydadzą się `pd.isna` lub metoda `isna`.

Polecenia wykonaj w terminalu z folderu practice. Mypy potrzebuje opisów typów pandas, więc raz dodaj pakiet pandas-stubs:

```bash
UV_HTTP_TIMEOUT=300 UV_HTTP_RETRIES=10 UV_CONCURRENT_DOWNLOADS=2 uv add --dev pandas-stubs
uv run ruff format features.py test_features.py
uv run ruff check features.py test_features.py
uv run mypy --disallow-untyped-defs features.py
uv run python -m pytest -q test_features.py
```

Oczekujesz `3 passed`. Pusty starter daje trzy niepowodzenia. Test, który porównuje wynik funkcji z tym samym wywołaniem funkcji, nie ma niezależnego oczekiwania.

## C11.1 · Dopisz nowe pliki do CI

Jeżeli w B7.2 skopiowano workflow do `.github/workflows/checks.yml`, dodaj w nim nowe pliki do istniejących kroków: `features.py test_features.py` obok `summary.py test_summary.py` w krokach ruff i mypy oraz `test_features.py` w kroku pytest. Zapisz zmianę razem z plikami features na gałęzi nauki; jeżeli workflow nie istnieje, pomiń go w git add:

```bash
git add practice/features.py practice/test_features.py practice/pyproject.toml practice/uv.lock .github/workflows/checks.yml
git diff --cached
git commit -m "Add series-aware lag feature with tests"
```

Przed commitem przeczytaj całą zmianę. Jeżeli nie wykonujesz części GitHub, lokalne sprawdzenia wystarczają.

## Sprawdź rezultat

Sprawdzenia notebooka przechodzą po restarcie kernela, a trzy testy przechodzą w terminalu. Do przemyślenia: dlaczego sortowanie nie odtworzy brakującego dnia 5 produktu apple? Czy lag1 dla apple w dniu 6 wolno nazwać „wczorajszą sprzedażą”? Co by się zmieniło w tabeli dostępności przy prognozie aktualizowanej codziennie? Nie musisz spisywać odpowiedzi.